In [6]:

import joblib
import pandas as pd

train_df = pd.read_csv('data/train.csv')
test_df = pd.read_csv('data/test.csv')

train_df['text'] = train_df['text'].fillna('')
test_df['text'] = test_df['text'].fillna('')

X_train, y_train = train_df['text'], train_df['source']
X_test, y_test = test_df['text'], test_df['source']

models = joblib.load('tuned_models.joblib')

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

positive_label = 1

results = []

for name, m in models.items():
    y_pred = m['model'].predict(X_test)

    results.append({
        'model': name,
        'params': m['params'],
        'cv_score': m['cv_score'],
        'test_accuracy': accuracy_score(y_test, y_pred),
        'test_precision': precision_score(y_test, y_pred, pos_label=positive_label),
        'test_recall': recall_score(y_test, y_pred, pos_label=positive_label),
        'test_f1': f1_score(y_test, y_pred, pos_label=positive_label),
    })

results_df = pd.DataFrame(results)
results_df.to_csv('test_results.csv', index=False)
results_df

,model,params,cv_score,test_accuracy,test_precision,test_recall,test_f1
0,naive_bayes,{'sel__k': 1750},0.908667,0.916,0.888060,0.952,0.918919
1,logistic_regression,{'logreg__C': 1.438449888287663},0.904667,0.916,0.922764,0.908,0.915323
2,random_forest,"{'max_features': 2, 'n_estimators': 600}",0.922000,0.936,0.982301,0.888,0.932773
3,decision_tree,{'tree__ccp_alpha': 0.0015873015873015873},0.792000,0.810,0.832618,0.776,0.803313
4,gradient_boosting,"{'gb__learning_rate': 0.5, 'gb__max_depth': 1}",0.911333,0.902,0.920502,0.880,0.899796


In [8]:
models.items()

dict_items([('naive_bayes', {'model': Pipeline(steps=[('vec',
                 CountVectorizer(min_df=5, ngram_range=(1, 2),
                                 token_pattern='\\b[^\\W\\d_]{2,}\\b')),
                ('sel',
                 SelectKBest(k=1750,
                             score_func=<function chi2 at 0x0000021EBA40C860>)),
                ('nb', MultinomialNB())]), 'params': {'sel__k': 1750}, 'cv_score': np.float64(0.9086666666666666)}), ('logistic_regression', {'model': Pipeline(steps=[('vec',
                 CountVectorizer(min_df=5, ngram_range=(1, 2),
                                 token_pattern='\\b[^\\W\\d_]{2,}\\b')),
                ('logreg',
                 LogisticRegression(C=np.float64(1.438449888287663), l1_ratio=1,
                                    random_state=42, solver='liblinear'))]), 'params': {'logreg__C': np.float64(1.438449888287663)}, 'cv_score': np.float64(0.9046666666666667)}), ('random_forest', {'model': Pipeline(steps=[('vec',
          

In [7]:
import numpy as np

def show(title, names, values, idx):
    print(title)
    for i in idx:
        print(f'   {names[i]:<25} {values[i]:.4f}')

# ---- Naive Bayes ----
pipe = models['naive_bayes']['model']
names = pipe.named_steps['vec'].get_feature_names_out()
names = names[pipe.named_steps['sel'].get_support()]   # keep only the selected k features
nb = pipe.named_steps['nb']

# log P(word | class 1) - log P(word | class 0)
diff = nb.feature_log_prob_[1] - nb.feature_log_prob_[0]
order = np.argsort(diff)

print('=== Naive Bayes ===')
show(f'Towards {nb.classes_[1]}:', names, diff, order[::-1][:5])
show(f'Towards {nb.classes_[0]}:', names, diff, order[:5])

# ---- Logistic regression ----
pipe = models['logistic_regression']['model']
names = pipe.named_steps['vec'].get_feature_names_out()
lr = pipe.named_steps['logreg']

coef = lr.coef_[0]   # positive -> class 1, negative -> class 0
order = np.argsort(coef)

print('\n=== Logistic regression ===')
show(f'Towards {lr.classes_[1]}:', names, coef, order[::-1][:5])
show(f'Towards {lr.classes_[0]}:', names, coef, order[:5])

# ---- Tree-based models (importance has no direction) ----
for model_name, step in [('decision_tree', 'tree'), ('random_forest', 'rf'), ('gradient_boosting', 'gb')]:
    pipe = models[model_name]['model']
    names = pipe.named_steps['vec'].get_feature_names_out()
    importance = pipe.named_steps[step].feature_importances_
    order = np.argsort(importance)[::-1][:5]

    print(f'\n=== {model_name} ===')
    show('Most important:', names, importance, order)

=== Naive Bayes ===
Towards 1:
   service was               3.5002
   connection was            3.3750
   attractions the           3.3296
   not up                    3.3060
   additionally the          3.1793
Towards 0:
   told                      -3.9870
   he                        -3.7989
   booked                    -3.7638
   said                      -3.6897
   want                      -3.6505

=== Logistic regression ===
Towards 1:
   service was               2.7245
   limited                   2.2665
   properly                  1.9474
   heart                     1.9099
   proximity                 1.8931
Towards 0:
   could not                 -2.2301
   roof                      -1.7492
   few                       -1.5187
   cleaning                  -1.3229
   breakfast and             -1.3073

=== decision_tree ===
Most important:
   rooms                     0.1907
   service                   0.1034
   room was                  0.0715
   we                        0

In [9]:
# %%
# EVALUATE FILE
# Computation of four performance measures (accuracy, precision, recall, F1 score)
# Run McNemar, proscribed signifigance test

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# Take true test labels, one model's predictions, and return a dictionary with 4 sklearn metrics.
# To be called once for each of the the five final models.
def evaluate(y_true, y_pred):
    # Metrics with AI-generated (label 1) as the positive class.
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, pos_label=1),
        "recall": recall_score(y_true, y_pred, pos_label=1),
        "f1": f1_score(y_true, y_pred, pos_label=1),
    }


# comparing models, seek disgreements in results, for questions 1 and 2 with alpha of .01
# both models scored on same 500 test reviews so results are paired
# use slide method of McNemar as an exact binomal test (slide 43 of log reg slides)
# Under H0 (equal accuracy), each disagreement is a fair coin flip, so the smaller
#   off-diagonal count ~ Binomial(n01 + n10, 0.5). Returns the 2x2 table
#   of correct/incorrect and the two-sided p-value.

def mcnemar(y_true, pred_a, pred_b, name_a="A", name_b="B"):
    # standardize as numpy arrays
    y_true, pred_a, pred_b = map(np.asarray, (y_true, pred_a, pred_b))
    a_ok, b_ok = pred_a == y_true, pred_b == y_true
    table = pd.DataFrame(
        [[np.sum(a_ok & b_ok), np.sum(a_ok & ~b_ok)],
         [np.sum(~a_ok & b_ok), np.sum(~a_ok & ~b_ok)]],
        # for dev testing
        index=[f"{name_a} correct", f"{name_a} wrong"],
        columns=[f"{name_b} correct", f"{name_b} wrong"])
    # strip labels, exact=true means binomial dist false is chi square approximation
    result = sm_mcnemar(table.values, exact=True)
    # include table in Overleaf, p-value to be compared to the alpha.
    return table, result.pvalue

# unpack via table, p = mcnemar(y_test, pred_nb, pred_lr, name_a="NB", name_b="LR")


In [10]:
from statsmodels.stats.contingency_tables import mcnemar as sm_mcnemar

pred_nb = models['naive_bayes']['model'].predict(X_test)
pred_lr = models['logistic_regression']['model'].predict(X_test)
pred_tree = models['decision_tree']['model'].predict(X_test)
pred_rf = models['random_forest']['model'].predict(X_test)
pred_gb = models['gradient_boosting']['model'].predict(X_test)

# Q1: generative vs discriminative linear model
table, p = mcnemar(y_test, pred_nb, pred_lr, 'NB', 'LR')
print(table, '\np =', p)

# Q2: ensembles vs single tree
table, p = mcnemar(y_test, pred_rf, pred_tree, 'RF', 'Tree')
print(table, '\np =', p)

table, p = mcnemar(y_test, pred_gb, pred_tree, 'GB', 'Tree')
print(table, '\np =', p)

            LR correct  LR wrong
NB correct         428        30
NB wrong            30        12 
p = 1.0
            Tree correct  Tree wrong
RF correct           394          74
RF wrong              11          21 
p = 1.2888315230163817e-12
            Tree correct  Tree wrong
GB correct           385          66
GB wrong              20          29 
p = 6.669778822829637e-07
